In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
import sys
import os

# Add the directory containing the file to the system path
sys.path.append(os.path.abspath("../module1"))

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [7]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

Running RAG

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

For this lesson, use RAGWithUsage from the evaluation utilities. It subclasses RAGBase from module 01, so it has the same rag method.

It stores token usage after each LLM call. Then we can calculate the total cost later.

It also uses the search boosts we selected in the search tuning lesson: question=1.0, answer=2.0, and section=0.1.

In [9]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

For each question, RAGBase searches the FAQ, builds a prompt with the retrieved context, and asks the LLM to answer. We save the answer so the next lesson can judge it.

Run RAG for one question:

In [10]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

Check the cost of this call:

In [12]:
assistant.total_cost()

0.00048300000000000003

Get the original answer from the document ID:



In [13]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

Now save both answers in one record:

In [14]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course, is it too late for me to join, or can I still sign up?',
 'answer_llm': 'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

Processing all questions

Create a function that processes one ground truth record:

In [15]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [16]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course, is it too late for me to join, or can I still sign up?',
 'answer_llm': 'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

Before running the full batch, reset the usage we collected while testing:

In [17]:
assistant.reset_usage()

This calls the LLM once per ground truth question, so it can take some time. Let's process the questions in parallel and track progress.

Import the parallel processing helper from the same utility file:



In [18]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

Run RAG for all ground truth questions:



In [19]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/565 [00:00<?, ?it/s]

Collect the answer records:

In [20]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [21]:
assistant.total_cost()

0.6174637500000005

In [22]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [23]:
df_answers[:5]

,question,answer_llm,answer_orig,document
0,"I just found this course, is it too late for m...","Yes, you can still join. You don’t need a conf...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,"If I join now, will I still be able to get a c...","Yes. If you join now, you can still get a cert...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,What do I need to do to qualify for the certif...,"To qualify for the certificate, you need to:\n...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Can I submit my final project after the submis...,No. You can only get certified if you finish t...,"Yes, but if you want to receive a certificate,...",74eb249bbf
4,Is there any deadline I should know about if I...,"Yes — if you want a certificate, you must subm...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [24]:
len(df_answers)

565